This notebook extracts remote sensing data at the reference data points for model building. 

In [ ]:
import os
import re
import glob
import geopandas as gpd
import pandas as pd
import rasterio
from rasterio.plot import show
from shapely.geometry import Point

In [ ]:
# set variables
COMBINED_DIR = "../data/grasslvnd/combined_2024"
TILES_FILE = "../data/grid_50km_epsg3059.geojson"
PTS_CSV = "../data/combined_sample_points_fin.csv"
OUT_CSV = "../data/combined_points_with_data_fin.csv"

In [ ]:
# add raster values to points
if PTS_CSV.lower().endswith((".shp", ".geojson", ".gpkg")):
    pts = gpd.read_file(PTS_CSV)
else:
    df = pd.read_csv(PTS_CSV)
    if not {"lon", "lat"}.issubset(df.columns):
        raise ValueError("need 'lon' and 'lat'.")
    pts = gpd.GeoDataFrame(df, geometry=gpd.points_from_xy(df.lon, df.lat), crs="EPSG:4326")


tiles = gpd.read_file(TILES_FILE)
if "grid_id" not in tiles.columns:
    raise ValueError("must have a 'grid_id' field.")
tiles = tiles.to_crs(epsg=3059)

pts_3059 = pts.to_crs(epsg=3059)
pts_3059 = gpd.sjoin(pts_3059, tiles[["grid_id", "geometry"]], how="left", predicate="within")
print(f"joined {len(pts_3059)} points to {tiles.shape[0]} tiles")

raster_files = glob.glob(os.path.join(COMBINED_DIR, "*.tif"))
if not raster_files:
    raise FileNotFoundError(f"No .tif rasters found in {COMBINED_DIR}")

# function to extract grid_id from raster filename
def extract_grid_id(filename):
    match = re.search(r"tile_(\d+)", os.path.basename(filename))
    return int(match.group(1)) if match else None

for raster_path in raster_files:
    grid_id = extract_grid_id(raster_path)
    if grid_id is None:
        print("no grid_id found.")
        continue

    pts_tile = pts_3059[pts_3059["grid_id"] == grid_id]
    if pts_tile.empty:
        continue

    with rasterio.open(raster_path) as src:
        coords = [(x, y) for x, y in zip(pts_tile.geometry.x, pts_tile.geometry.y)]
        samples = list(src.sample(coords))
        band_names = [src.descriptions[b - 1] if src.descriptions[b - 1] else f"Band_{b}" 
                      for b in range(1, src.count + 1)]

        for idx, vals in zip(pts_tile.index, samples):
            for bname, v in zip(band_names, vals):
                pts_3059.loc[idx, bname] = v


In [ ]:
# export results
pts_out = pts_3059.to_crs(epsg=4326)
drop_cols = ["geometry", "index_right"]
pts_out.drop(columns=[c for c in drop_cols if c in pts_out.columns], inplace=True)
pts_out.to_csv(OUT_CSV, index=False)